# Invasive (INV): OME-TIFF -> SpatialData -> precomputed raster + meshes -> Neuroglancer

Same pipeline structure as melanoma. This dataset has ~44,000 real segmented cells with confirmed `PhysicalSizeX=PhysicalSizeY=1.0 µm`

In [ ]:
%load_ext jupyter_black

## 1. Setup

In [ ]:
from pathlib import Path
from spatialdata import SpatialData
from spatialdata.models import Labels3DModel
from dask_image.imread import imread
import xmltodict
import tifffile

from tissue_map_tools.igneous_converters import (
    from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes,
)
from tissue_map_tools.view import view_precomputed_in_neuroglancer
from tissue_map_tools.utils import compute_initial_camera_state

dataset_path = Path.cwd().parent.parent / "data" / "invasive"
raw_path = dataset_path / "raw"
out_path = dataset_path / "out"
raw_path.mkdir(parents=True, exist_ok=True)
out_path.mkdir(parents=True, exist_ok=True)
ome_tiff_path = raw_path / "invasive_mask.ome.tiff"
precomputed_path = out_path / "invasive_precomputed"

if not ome_tiff_path.exists():
    raise FileNotFoundError(
        f"{ome_tiff_path} does not exist. Please use symlinks to make the data available."
    )

## 2. Load and determine axis order (same heuristic and same caveat as melanoma)

In [ ]:
data = imread(ome_tiff_path)

xml = tifffile.TiffFile(ome_tiff_path).ome_metadata
xml_dict = xmltodict.parse(xml)
sizes = {
    ax: int(xml_dict["OME"]["Image"]["Pixels"][f"@Size{ax.upper()}"]) for ax in "xyz"
}
assert len(set(sizes.values())) == 3, (
    "Sizes collide -- do not trust the heuristic below."
)

dims = []
for size in data.shape:
    for ax, ax_size in sizes.items():
        if size == ax_size:
            dims.append(ax)
            break
dims = tuple(dims)

print("data.shape =", data.shape)
print("computed dims =", dims)

## 3. Build the SpatialData object

In [ ]:
if not (precomputed_path / "info").exists():
    labels = Labels3DModel.parse(data, dims=dims)
    sdata_write_path = out_path / "invasive_mask.zarr"

    sdata_unwritten = SpatialData.init_from_elements({"labels": labels})
    sdata_unwritten.write(str(sdata_write_path), overwrite=True)

    sdata = SpatialData.read(str(sdata_write_path))

## 4. Convert to precomputed raster + meshes

In [ ]:
%%time
if not (precomputed_path / "info").exists():
    from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes(
        raster=sdata["labels"],
        precomputed_path=str(precomputed_path),
        shape=(128, 128, 128),
        nlod=3,
        min_chunk_size=(32, 32, 32),
    )
    print("Conversion complete.")
else:
    print("Precomputed output already exists -- skipping conversion.")

## 5. Visualize in Neuroglancer


In [ ]:
# viewer = view_precomputed_in_neuroglancer(data_path=str(precomputed_path))
# viewer

## 6. Visualize in Vitessce
To view the data in Vitessce.
`use_web_app=True` opens vitessce.io in a new browser tab — note this blocks the notebook cell, waiting for you to press Enter in the running kernel's terminal once you're done viewing, before the cell completes.
`use_web_app=False` (or if not provided -- default) opens the Vitessce widget inline in this notebook.

When `segments` and `obsSets.csv` are not provided, all segments from the meshes will be visualized as obsSets.

**Note**: If the loading overlay persists on subsequent runs, perform a clean restart by shutting down all active kernels.

In [ ]:
# Stop previously started local servers before re-running
import vitessce
vitessce.data_server.stop_all()

In [ ]:
from tissue_map_tools.utils import compute_initial_camera_state
from tissue_map_tools.vitessce_configs.layer_specs import SegmentationLayerSpec
from tissue_map_tools.vitessce_configs.neuroglancer_config_builder import build_neuroglancer_config

# segments = ["612", "3351", "4328", "6531", "8446"]
use_web_app = True


initial_camera_state = compute_initial_camera_state(
    data_path=str(precomputed_path),
    # segments=segments,
      # camera_segments=segments
)

vc = build_neuroglancer_config(
    name="Precomputed data",
    segmentations=[
        SegmentationLayerSpec(
            file_uid="segmentation", 
            data_path=str(precomputed_path), 
            # segments=segments
        ),
    ],
  
    initial_camera_state=initial_camera_state,
    use_web_app = use_web_app
)

vc